# SWYNEX Technologies — Task 1: Clinical Data Preparation
### Project: Real-World Clinical Data Preprocessing & Profiling
**Author:** Abhijeet Arun Gore  
**Role:** Data Science Intern at SWYNEX Technologies  
**Dataset:** Diabetes 130-US Hospitals for Years 1999–2008 (UCI Machine Learning Repository)  

---

## 1. Dataset Introduction
This project focuses on **Task 1: Data Preparation** for the *Diabetes 130-US Hospitals dataset (1999–2008)* sourced from the **UCI Machine Learning Repository**. The dataset comprises **101,766 clinical hospital encounters** across 130 US hospitals and contains **50 clinical, demographic, and administrative features** describing inpatient admissions of diabetic patients.


In [ ]:
import os
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'Segoe UI'
plt.rcParams['figure.dpi'] = 120

# Set display options for thorough inspection
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

print("Libraries and global configurations initialized successfully.")


## 2. Load Dataset
We load the raw dataset (`diabetic_data.csv`) and the lookup mapping file (`IDS_mapping.csv`) stored in `data/raw/`.

In [ ]:
raw_data_path = os.path.join('data', 'raw', 'diabetic_data.csv')
ids_mapping_path = os.path.join('data', 'raw', 'IDS_mapping.csv')

df_raw = pd.read_csv(raw_data_path)
print(f"Raw Dataset loaded successfully. Dimensions: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

with open(ids_mapping_path, 'r') as f:
    ids_raw_text = f.read()

print("IDS Mapping lookup table loaded successfully.")


## 3. Initial Data Inspection
We execute standard diagnostic operations to profile feature dimensions, data types, and missing value indicators.

In [ ]:
print("=== DATASET DIMENSIONS & DATA TYPES ===")
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")
print("\n=== DATASET INFO SUMMARY ===")
df_raw.info(verbose=True, show_counts=True)


In [ ]:
print('=== RAW DATASET HEAD (FIRST 5 ROWS) ===')
df_raw.head()

In [ ]:
print('=== RAW DATASET TAIL (LAST 5 ROWS) ===')
df_raw.tail()

In [ ]:
print('=== DESCRIPTIVE STATISTICS (ALL 50 FEATURES) ===')
df_raw.describe(include='all').T

## 4. Missing-Value Analysis
Real-world healthcare datasets frequently utilize placeholder characters such as `'?'` or `'Unknown/Invalid'` to indicate missing measurements. In this section, we replace implicit missing indicators with standard `np.nan` values and quantify missingness across all features.


In [ ]:
df_proc = df_raw.copy()

missing_placeholder_cols = ['race', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3']
for col in missing_placeholder_cols:
    df_proc[col] = df_proc[col].replace('?', np.nan)

df_proc['gender'] = df_proc['gender'].replace('Unknown/Invalid', np.nan)


In [ ]:
# Lab test results 'None' means 'Not Tested'
df_proc['max_glu_serum'] = df_proc['max_glu_serum'].fillna('Not Tested').replace('None', 'Not Tested')
df_proc['A1Cresult'] = df_proc['A1Cresult'].fillna('Not Tested').replace('None', 'Not Tested')


In [ ]:
missing_counts = df_proc.isnull().sum()
missing_percentages = (missing_counts / len(df_proc)) * 100

missing_df = pd.DataFrame({
    'Feature': missing_counts.index,
    'Missing_Count': missing_counts.values,
    'Missing_Percentage': missing_percentages.values
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False).reset_index(drop=True)

print("=== FEATURES CONTAINING MISSING VALUES ===")
missing_df


In [ ]:
plt.figure(figsize=(10, 5))
barplot = sns.barplot(
    data=missing_df,
    x='Missing_Percentage',
    y='Feature',
    palette='Reds_r'
)
plt.title('Missing Value Percentage by Clinical Feature', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Missing Percentage (%)', fontsize=12)
plt.ylabel('Clinical Feature', fontsize=12)

for p in barplot.patches:
    width = p.get_width()
    plt.text(width + 1.2, p.get_y() + p.get_height()/2, f'{width:.2f}%', 
             ha='left', va='center', fontsize=10, fontweight='bold')

plt.xlim(0, 110)
plt.tight_layout()
plt.show()


## 5. Duplicate Analysis
We check for exact row duplicates, unique encounter identifiers (`encounter_id`), and patient admission frequencies (`patient_nbr`).

In [ ]:
print("=== DUPLICATE ANALYSIS RESULTS ===")
exact_dups = df_proc.duplicated().sum()
encounter_dups = df_proc['encounter_id'].duplicated().sum()
patient_dups = df_proc['patient_nbr'].duplicated().sum()

print(f"1. Exact Row Duplicates: {exact_dups}")
print(f"2. Duplicate Encounter IDs: {encounter_dups}")
print(f"3. Duplicate Patient IDs (Repeat Admissions): {patient_dups:,} ({(patient_dups/len(df_proc))*100:.2f}%)")

patient_counts = df_proc['patient_nbr'].value_counts()
print(f"\nTotal Distinct Patients: {len(patient_counts):,}")
print(f"Single-Encounter Patients: {(patient_counts == 1).sum():,} ({((patient_counts == 1).sum()/len(patient_counts))*100:.2f}%)")
print(f"Multi-Encounter Patients: {(patient_counts > 1).sum():,} ({((patient_counts > 1).sum()/len(patient_counts))*100:.2f}%)")


## 6. Data-Type Handling
We map relational lookup tables from `IDS_mapping.csv` to replace numeric IDs with descriptive clinical text, and drop zero-variance features.

In [ ]:
lines = ids_raw_text.split('\n')
mapping_dicts = {'admission_type_id': {}, 'discharge_disposition_id': {}, 'admission_source_id': {}}
current_key = None

for line in lines:
    line_str = line.strip()
    if not line_str or line_str == ',':
        continue
    if line_str.startswith('admission_type_id'):
        current_key = 'admission_type_id'
        continue
    elif line_str.startswith('discharge_disposition_id'):
        current_key = 'discharge_disposition_id'
        continue
    elif line_str.startswith('admission_source_id'):
        current_key = 'admission_source_id'
        continue
        
    parts = line_str.split(',', 1)
    if len(parts) == 2 and parts[0].isdigit():
        v_id = int(parts[0])
        desc = parts[1].strip('"').strip()
        if desc.upper() in ['NULL', 'NAN', 'NONE', '']:
            desc = 'Not Available / Unmapped'
        mapping_dicts[current_key][v_id] = desc

df_proc['admission_type_desc'] = df_proc['admission_type_id'].map(mapping_dicts['admission_type_id']).fillna('Not Available / Unmapped').replace('NULL', 'Not Available / Unmapped')
df_proc['discharge_disposition_desc'] = df_proc['discharge_disposition_id'].map(mapping_dicts['discharge_disposition_id']).fillna('Not Available / Unmapped').replace('NULL', 'Not Available / Unmapped')
df_proc['admission_source_desc'] = df_proc['admission_source_id'].map(mapping_dicts['admission_source_id']).fillna('Not Available / Unmapped').replace('NULL', 'Not Available / Unmapped')

print("=== RELATIONAL ID MAPPING COMPLETED ===")
df_proc[['admission_type_id', 'admission_type_desc', 'discharge_disposition_id', 'discharge_disposition_desc']].head(5)


In [ ]:
zero_var_cols = [col for col in df_proc.columns if df_proc[col].nunique(dropna=False) == 1]
print(f"Zero Variance Features Identified: {zero_var_cols}")


In [ ]:
df_proc = df_proc.drop(columns=zero_var_cols + ['weight'])
print(f"Shape after dropping zero-variance features and weight: {df_proc.shape}")


## 7. Categorical Standardization
High-cardinality features and unstandardized categorical text increase feature space complexity. Here we convert age brackets to numerical midpoints and map 900+ ICD-9 diagnosis codes into 9 primary clinical categories.

In [ ]:
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35, '[40-50)': 45,
    '[50-60)': 55, '[60-70)': 65, '[70-80)': 75, '[80-90)': 85, '[90-100)': 95
}
df_proc['age_midpoint'] = df_proc['age'].map(age_map).astype(float)


In [ ]:
def map_icd9_category(code):
    if pd.isna(code) or code == '?' or code == 'Unknown':
        return 'Missing/Unknown'
    code_str = str(code).strip()
    if code_str.startswith('V') or code_str.startswith('E'):
        return 'Other'
    try:
        val = float(code_str)
    except ValueError:
        return 'Other'
    
    if (390 <= val <= 459) or val == 785:
        return 'Circulatory'
    elif (460 <= val <= 519) or val == 786:
        return 'Respiratory'
    elif (520 <= val <= 579) or val == 787:
        return 'Digestive'
    elif int(val) == 250:
        return 'Diabetes'
    elif 800 <= val <= 999:
        return 'Injury'
    elif 710 <= val <= 739:
        return 'Musculoskeletal'
    elif (580 <= val <= 629) or val == 788:
        return 'Genitourinary'
    elif 140 <= val <= 239:
        return 'Neoplasms'
    else:
        return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df_proc[f'{col}_category'] = df_proc[col].apply(map_icd9_category)

print("=== ICD-9 DIAGNOSIS CATEGORIZATION SAMPLE (diag_1_category) ===")
print(df_proc['diag_1_category'].value_counts())


## 8. Missing-Value Treatment
We apply context-specific imputation strategies grounded in clinical data science best practices.

In [ ]:
df_clean = df_proc.copy()
df_clean = df_clean.dropna(subset=['gender'])
